In [1]:
import pandas as pd
import sys
import numpy as np

In [2]:
sys.path.append("..")

In [3]:
from src.data_loader import load_data, save_data_to_csv

In [4]:
data = load_data("QQQ", "2010-01-01", "2026-06-23")

[*********************100%***********************]  1 of 1 completed


In [6]:
start = data.index[0].year
end = data.index[-1].year

end- start

16

In [7]:
start_year = data.index[0].year

train_window = 5
forecast_window = 1

for i in range((data.index[-1].year-data.index[0].year)-train_window + 1):

    train_start = start_year
    train_end = start_year + train_window

    forecast_start = train_end
    forecast_end = forecast_start + forecast_window

    train_data = data[
        (data.index.year >= train_start) &
        (data.index.year < train_end)
    ]

    forecast_data = data[
        (data.index.year >= forecast_start) &
        (data.index.year < forecast_end)
    ]

    print(
        f"Train: {train_start}-{train_end-1}"
    )

    print(
        f"Forecast: {forecast_start}"
    )

    start_year += 1

Train: 2010-2014
Forecast: 2015
Train: 2011-2015
Forecast: 2016
Train: 2012-2016
Forecast: 2017
Train: 2013-2017
Forecast: 2018
Train: 2014-2018
Forecast: 2019
Train: 2015-2019
Forecast: 2020
Train: 2016-2020
Forecast: 2021
Train: 2017-2021
Forecast: 2022
Train: 2018-2022
Forecast: 2023
Train: 2019-2023
Forecast: 2024
Train: 2020-2024
Forecast: 2025
Train: 2021-2025
Forecast: 2026


In [8]:
save_data_to_csv(data, "../data/QQQ_data.csv")

In [9]:
data

Price,Close,High,Low,Open,Volume
Date,,,,,
2010-01-04,40.246567,40.307260,40.116517,40.168539,62822800
2010-01-05,40.246567,40.315929,40.021146,40.220558,62935600
2010-01-06,40.003784,40.359258,39.943094,40.229208,96033000
2010-01-07,40.029797,40.116499,39.813044,40.064478,77094100
2010-01-08,40.359264,40.359264,39.821720,39.943100,88886600
...,...,...,...,...,...
2026-06-15,743.183289,743.942464,736.570560,737.289741,46710200
2026-06-16,729.058777,743.402998,728.839048,741.435190,45348700
2026-06-17,721.716858,734.872383,720.058646,734.382931,51669300


In [10]:
data = data[["Close"]]
data["log_returns"] = np.log(data).diff()

In [11]:
from models.garch import fit_garch_1_1, forecast_garch_1_1_volatility
from models.har import fit_har_rv, forecast_har_rv_volatility 

In [12]:
a = fit_garch_1_1(data[["log_returns"]])

In [13]:
result = forecast_garch_1_1_volatility(a)

In [14]:
result

np.float64(0.01840306436530191)

In [15]:
b = fit_har_rv(data[["log_returns"]])

In [16]:
har_result = forecast_har_rv_volatility(b)

In [17]:
har_result

0.016414464657420188

In [18]:
from models.lstm import create_sequences, fit_lstm, forecast_lstm_volatility

In [19]:
data_train = data[data.index < "2020-01-01"]["log_returns"]

In [20]:
model = fit_lstm(data_train)

In [21]:
model

LSTMVolatilityModel(
  (lstm): LSTM(1, 10, batch_first=True)
  (fc): Linear(in_features=10, out_features=1, bias=True)
)

In [22]:
test_window = data["log_returns"].dropna().iloc[-30:]

forecast = forecast_lstm_volatility(
    model,
    test_window
)

print(forecast)

0.0040331005428014335


In [23]:
from models import egarch, garch, gjr_garch, gru, har, lstm

In [24]:
models_df = pd.DataFrame({
    "model": [
        "GARCH",
        "EGARCH",
        "GJR-GARCH",
        "HAR",
        "LSTM",
        "GRU"
    ],
    "fit_function": [
        garch.fit_garch_1_1,
        egarch.fit_egarch_1_1,
        gjr_garch.fit_gjrgarch_1_1,
        har.fit_har_rv,
        lstm.fit_lstm,
        gru.fit_gru
    ],
    "forecast_function": [
        garch.forecast_garch_1_1_volatility,
        egarch.forecast_egarch_1_1_volatility,
        gjr_garch.forecast_gjrgarch_1_1_volatility,
        har.forecast_har_rv_volatility,
        lstm.forecast_lstm_volatility,
        gru.forecast_gru_volatility
    ]
})

In [25]:
models_df

,model,fit_function,forecast_function
0,GARCH,<function fit_garch_1_1 at 0x0000023837CE3420>,<function forecast_garch_1_1_volatility at 0x0...
1,EGARCH,<function fit_egarch_1_1 at 0x000002385B85E520>,<function forecast_egarch_1_1_volatility at 0x...
2,GJR-GARCH,<function fit_gjrgarch_1_1 at 0x000002385B85EAC0>,<function forecast_gjrgarch_1_1_volatility at ...
3,HAR,<function fit_har_rv at 0x000002384A36FCE0>,<function forecast_har_rv_volatility at 0x0000...
4,LSTM,<function fit_lstm at 0x000002385714D9E0>,<function forecast_lstm_volatility at 0x000002...
5,GRU,<function fit_gru at 0x000002385B83B600>,<function forecast_gru_volatility at 0x0000023...


In [5]:
import sys
sys.path.append("../src")

In [6]:
from training import train_models

In [7]:
result = train_models(data)

Train: 2010-2014
Forecast: 2015
Train: 2011-2015
Forecast: 2016
Train: 2012-2016
Forecast: 2017
Train: 2013-2017
Forecast: 2018
Train: 2014-2018
Forecast: 2019
Train: 2015-2019
Forecast: 2020
Train: 2016-2020
Forecast: 2021
Train: 2017-2021
Forecast: 2022
Train: 2018-2022
Forecast: 2023
Train: 2019-2023
Forecast: 2024
Train: 2020-2024
Forecast: 2025
Train: 2021-2025
Forecast: 2026


In [8]:
result

model,EGARCH,GARCH,GJR-GARCH,GRU,HAR,LSTM
date,,,,,,
2015-01-02,0.009869,0.009674,0.009863,0.004301,0.008770,0.003820
2015-01-05,0.010007,0.009722,0.009895,0.004121,0.009155,0.003681
2015-01-06,0.010154,0.009767,0.009871,0.004572,0.009355,0.004052
2015-01-07,0.010296,0.009811,0.009868,0.004890,0.009816,0.004351
2015-01-08,0.010413,0.009853,0.009780,0.005115,0.010098,0.004663
...,...,...,...,...,...,...
2026-06-15,0.015429,0.014126,0.012809,0.006852,0.014208,0.007616
2026-06-16,0.015426,0.014135,0.012788,0.007016,0.014208,0.008731
2026-06-17,0.015455,0.014144,0.012743,0.007206,0.014209,0.009266


In [ ]:
log_returns = utils.calculate_log_returns(data)

log_returns["log_returns"].abs().describe()